In [1]:
!pip install -q chromadb sentence-transformers llama-index llama-index-vector-stores-chroma datasets --upgrade
!pip install -q gradio

import requests
import json
import time

def search_papers(keyword, limit=50):
  papers = []
  url = "https://api.semanticscholar.org/graph/v1/paper/search"
  params = {
      "query": keyword,
      "limit": min(limit, 100),
      "fields": "title,abstract,year,authors,url,venue"
  }
  resp = requests.get(url, params=params)
  if resp.status_code == 200:
      papers = resp.json().get("data", [])
  time.sleep(1.2)
  return papers

keywords = [
  "machine translation neural network",
  "large language model fine-tuning efficient",
  "retrieval augmented generation NLP",
  "low-resource machine translation",
  "text summarization abstractive",
  "multilingual NLP cross-lingual transfer",
  "RLHF language model alignment",
  "named entity recognition few-shot",
]

all_papers = []
for kw in keywords:
  papers = search_papers(kw, limit=40)
  all_papers.extend(papers)
  print(f"'{kw}': {len(papers)} 篇")

# 去重
seen = set()
unique_papers = []
for p in all_papers:
  pid = p.get("paperId", "")
  if pid and pid not in seen and p.get("abstract"):
      seen.add(pid)
      unique_papers.append(p)

print(f"\n去重后: {len(unique_papers)} 篇论文")

with open('/kaggle/working/nlp_papers.json', 'w', encoding='utf-8') as f:
  json.dump(unique_papers, f, ensure_ascii=False, indent=2)

print("已保存到 nlp_papers.json")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 75.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 106.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 100.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 84.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━

In [2]:
import requests, json, time

# 加载已有论文
with open('/kaggle/working/nlp_papers.json', 'r', encoding='utf-8') as f:
  all_papers = json.load(f)

# 补充搜索——用更短的关键词
extra_keywords = [
  "machine translation",
  "language model fine tuning",
  "retrieval augmented generation",
  "named entity recognition",
  "question answering NLP",
  "text generation evaluation",
  "knowledge distillation NLP",
  "cross lingual representation",
]

for kw in extra_keywords:
  url = "https://api.semanticscholar.org/graph/v1/paper/search"
  params = {"query": kw, "limit": 20, "fields": "title,abstract,year,authors,url,venue"}
  resp = requests.get(url, params=params)
  if resp.status_code == 200:
      papers = resp.json().get("data", [])
      all_papers.extend(papers)
      print(f"'{kw}': {len(papers)} 篇")
  time.sleep(1.2)

# 去重
seen = set()
unique = []
for p in all_papers:
  pid = p.get("paperId", "")
  if pid and pid not in seen and p.get("abstract"):
      seen.add(pid)
      unique.append(p)

print(f"\n总计: {len(unique)} 篇论文")

with open('/kaggle/working/nlp_papers.json', 'w', encoding='utf-8') as f:
  json.dump(unique, f, ensure_ascii=False, indent=2)


总计: 100 篇论文


In [4]:
!pip install -q llama-index-embeddings-huggingface

In [5]:
from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb
import json

# 加载论文
with open('/kaggle/working/nlp_papers.json', 'r', encoding='utf-8') as f:
  papers = json.load(f)

print(f"加载 {len(papers)} 篇论文")

# 构造成 Document
documents = []
for p in papers:
  text = (
      f"Title: {p['title']}\n"
      f"Abstract: {p.get('abstract', '')}\n"
      f"Year: {p.get('year', 'N/A')}\n"
      f"Venue: {p.get('venue', 'N/A')}"
  )
  doc = Document(
      text=text,
      metadata={
          "title": p['title'],
          "year": str(p.get('year', '')),
          "venue": p.get('venue', ''),
          "url": p.get('url', ''),
      }
  )
  documents.append(doc)

# 分块
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=64)
nodes = splitter.get_nodes_from_documents(documents)
print(f"{len(documents)} 篇论文 → {len(nodes)} 个 chunks")

# Embedding 模型（384维，轻量快速）
embed_model = HuggingFaceEmbedding(
  model_name="BAAI/bge-small-zh",
  device="cuda"
)

# ChromaDB 向量数据库
chroma_client = chromadb.PersistentClient(path="/kaggle/working/chroma_db")
chroma_collection = chroma_client.get_or_create_collection("nlp_papers")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# 构建索引
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex(nodes, embed_model=embed_model, storage_context=storage_context)

print(f"索引构建完成！向量维度: {embed_model._model.get_sentence_embedding_dimension()}")
print("索引已持久化")

加载 100 篇论文


100 篇论文 → 100 个 chunks


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

索引构建完成！向量维度: 512
索引已持久化


/tmp/ipykernel_58/969274341.py:54: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"索引构建完成！向量维度: {embed_model._model.get_sentence_embedding_dimension()}")


In [6]:
retriever = index.as_retriever(similarity_top_k=5)

test_queries = [
  "How to handle rare words in neural machine translation?",
  "How does RLHF improve LLM alignment?",
  "What are the main challenges in multilingual NLP?",
  "How to evaluate text summarization quality?",
]

for query in test_queries:
  print(f"\n{'='*60}")
  print(f"查询: {query}")

  results = retriever.retrieve(query)
  for i, node in enumerate(results):
      print(f"\n  Top-{i+1} (score: {node.score:.4f})")
      print(f"  标题: {node.metadata.get('title', 'N/A')[:80]}")
      print(f"  年份: {node.metadata.get('year', 'N/A')}")


查询: How to handle rare words in neural machine translation?

  Top-1 (score: 0.5822)
  标题: Abstractive Text Summarization using Pre-Trained Language Model "Text-to-Text Tr
  年份: 2023

  Top-2 (score: 0.5756)
  标题: Abstractive Text Summarization using Sequence-to-sequence RNNs and Beyond
  年份: 2016

  Top-3 (score: 0.5731)
  标题: Abstractive Text Summarization Using the BRIO Training Paradigm
  年份: 2023

  Top-4 (score: 0.5700)
  标题: Neural Attention Model for Abstractive Text Summarization Using Linguistic Featu
  年份: 2023

  Top-5 (score: 0.5694)
  标题: Bidirectional and Auto-Regressive Transformer (BART) for Indonesian Abstractive 
  年份: 2024

查询: How does RLHF improve LLM alignment?

  Top-1 (score: 0.5309)
  标题: InfAlign: Inference-aware language model alignment
  年份: 2024

  Top-2 (score: 0.5287)
  标题: AlignDistil: Token-Level Language Model Alignment as Adaptive Policy Distillatio
  年份: 2025

  Top-3 (score: 0.5281)
  标题: MM-RLHF: The Next Step Forward in Multimodal LLM Alignment


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
  MODEL_NAME,
  torch_dtype=torch.float16,
  trust_remote_code=True,
  device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def ask_nlp_question(query):
  # 检索相关论文
  results = retriever.retrieve(query)

  # 拼接上下文
  context_parts = []
  for i, node in enumerate(results):
      title = node.metadata.get('title', 'N/A')
      year = node.metadata.get('year', 'N/A')
      context_parts.append(f"[{i+1}] {title} ({year})\n{node.text[:300]}")
  context = "\n\n".join(context_parts)

  # 构造 prompt
  prompt = f"""You are an NLP research assistant. Answer the question based on the retrieved papers below.
If the papers don't provide enough information, say so honestly. Do not fabricate.

Retrieved papers:
{context}

Question: {query}

Answer:"""

  messages = [
      {"role": "user", "content": prompt},
  ]
  formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

  with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=200,
          do_sample=True,
          temperature=0.3,
          top_p=0.9,
          pad_token_id=tokenizer.pad_token_id,
      )

  response = tokenizer.decode(outputs[0], skip_special_tokens=True)
  if "assistant" in response:
      response = response.split("assistant")[-1].strip()

  return response, results

# 测试
test_q = "How does RLHF improve LLM alignment?"
answer, sources = ask_nlp_question(test_q)
print(f"Q: {test_q}")
print(f"A: {answer}")
print(f"\n引用论文:")
for i, node in enumerate(sources):
  print(f"  [{i+1}] {node.metadata.get('title', 'N/A')[:60]} ({node.metadata.get('year', '')})")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q: How does RLHF improve LLM alignment?
A: The paper "MM-RLHF" proposes a new method called "MM-RLHF," which aims to enhance LLM alignment by incorporating reinforcement learning from human feedback (RLHF). Specifically, this method uses RLHF to ensure that large language models (LLMs) align with human values. By leveraging RLHF, the authors aim to improve the alignment process and reduce errors in the alignment results. This approach addresses the challenges faced by traditional RLHF methods, making it more effective at aligning large language models with human preferences.

引用论文:
  [1] InfAlign: Inference-aware language model alignment (2024)
  [2] AlignDistil: Token-Level Language Model Alignment as Adaptiv (2025)
  [3] MM-RLHF: The Next Step Forward in Multimodal LLM Alignment (2025)
  [4] Proxy-RLHF: Decoupling Generation and Alignment in Large Lan (2024)
  [5] Accelerated Preference Optimization for Large Language Model (2024)


In [8]:
import gradio as gr

def rag_translate(query):
  if not query.strip():
      return "Please enter a question."
  answer, sources = ask_nlp_question(query)

  # 拼接引用
  refs = "\n\n**References:**\n"
  for i, node in enumerate(sources):
      title = node.metadata.get('title', 'N/A')
      year = node.metadata.get('year', '')
      refs += f"\n[{i+1}] {title} ({year})"

  return answer + refs

demo = gr.Interface(
  fn=rag_translate,
  inputs=gr.Textbox(lines=3, placeholder="Ask an NLP question...", label="Question"),
  outputs=gr.Textbox(lines=15, label="Answer with References"),
  title="NLP Literature Q&A (RAG)",
  description="Ask questions about NLP research. Powered by 100+ papers + Qwen2.5 + RAG.",
  examples=[
      ["How does RLHF improve LLM alignment?"],
      ["What are the main challenges in multilingual NLP?"],
      ["How to evaluate text summarization quality?"],
  ],
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://28ca182464f8b699e6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb, json

with open('/kaggle/working/nlp_papers.json', 'r', encoding='utf-8') as f:
  papers = json.load(f)

documents = []
for p in papers:
  text = f"Title: {p['title']}\nAbstract: {p.get('abstract', '')}"
  doc = Document(text=text, metadata={"title": p['title'], "year": str(p.get('year', ''))})
  documents.append(doc)

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-zh", device="cuda")
chroma_client = chromadb.PersistentClient(path="/kaggle/working/chroma_db_exp")

results_summary = {}
test_query = "How to improve low-resource machine translation?"

for chunk_size in [256, 512, 1024]:
  print(f"\n--- Chunk Size = {chunk_size} ---")

  splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=chunk_size // 8)
  nodes = splitter.get_nodes_from_documents(documents)
  print(f"  Chunks: {len(nodes)}")

  collection_name = f"nlp_cs_{chunk_size}"
  try:
      chroma_client.delete_collection(collection_name)
  except:
      pass
  collection = chroma_client.get_or_create_collection(collection_name)
  vector_store = ChromaVectorStore(chroma_collection=collection)

  storage_context = StorageContext.from_defaults(vector_store=vector_store)
  idx = VectorStoreIndex(nodes, embed_model=embed_model, storage_context=storage_context)

  retriever = idx.as_retriever(similarity_top_k=5)
  results = retriever.retrieve(test_query)

  scores = [r.score for r in results]
  titles = [r.metadata.get('title', '')[:50] for r in results]
  print(f"  Top-1: {scores[0]:.4f} | {titles[0]}")
  print(f"  Top-5 Avg: {sum(scores)/len(scores):.4f}")
  results_summary[chunk_size] = scores

print("\n" + "="*50)
print("Chunk Size 对比表:")
print("| Chunk Size | Chunks | Top-1 Score | Top-5 Avg |")
print("|---|---|---|---|")
for cs, scores in results_summary.items():
  print(f"| {cs} | - | {scores[0]:.4f} | {sum(scores)/len(scores):.4f} |")

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Chunk Size = 256 ---
  Chunks: 159
  Top-1: 0.5860 | CUTE: A Multilingual Dataset for Enhancing Cross-L
  Top-5 Avg: 0.5775

--- Chunk Size = 512 ---
  Chunks: 100
  Top-1: 0.5604 | Why Low-Resource NLP Needs More Than Cross-Lingual
  Top-5 Avg: 0.5550

--- Chunk Size = 1024 ---
  Chunks: 100
  Top-1: 0.5604 | Why Low-Resource NLP Needs More Than Cross-Lingual
  Top-5 Avg: 0.5550

Chunk Size 对比表:
| Chunk Size | Chunks | Top-1 Score | Top-5 Avg |
|---|---|---|---|
| 256 | - | 0.5860 | 0.5775 |
| 512 | - | 0.5604 | 0.5550 |
| 1024 | - | 0.5604 | 0.5550 |


In [10]:
from llama_index.core.postprocessor import SentenceTransformerRerank

# 用之前的 512 chunk 索引
retriever = index.as_retriever(similarity_top_k=10)

# Reranker
reranker = SentenceTransformerRerank(
  model="BAAI/bge-reranker-base",
  top_n=3,
)

test_queries = [
  "How does RLHF improve LLM alignment?",
  "What are challenges in low-resource machine translation?",
]

for q in test_queries:
  print(f"\n{'='*60}")
  print(f"查询: {q}")

  # 无 Reranker
  results_no = retriever.retrieve(q)
  print("\n--- 无 Reranker (Top-3 of 10) ---")
  for i, node in enumerate(results_no[:3]):
      print(f"  [{i+1}] score={node.score:.4f} | {node.metadata.get('title','')[:60]}")

  # 有 Reranker
  results_rerank = reranker.postprocess_nodes(results_no, query_str=q)
  print("\n--- 有 Reranker (Top-3 of 10) ---")
  for i, node in enumerate(results_rerank[:3]):
      print(f"  [{i+1}] score={node.score:.4f} | {node.metadata.get('title','')[:60]}")

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]


查询: How does RLHF improve LLM alignment?

--- 无 Reranker (Top-3 of 10) ---
  [1] score=0.5309 | InfAlign: Inference-aware language model alignment
  [2] score=0.5287 | AlignDistil: Token-Level Language Model Alignment as Adaptiv
  [3] score=0.5281 | MM-RLHF: The Next Step Forward in Multimodal LLM Alignment

--- 有 Reranker (Top-3 of 10) ---
  [1] score=0.9670 | Accelerated Preference Optimization for Large Language Model
  [2] score=0.9653 | MM-RLHF: The Next Step Forward in Multimodal LLM Alignment
  [3] score=0.9283 | Proxy-RLHF: Decoupling Generation and Alignment in Large Lan

查询: What are challenges in low-resource machine translation?

--- 无 Reranker (Top-3 of 10) ---
  [1] score=0.6127 | Why Low-Resource NLP Needs More Than Cross-Lingual Transfer:
  [2] score=0.5951 | Abstractive text summarization of low-resourced languages us
  [3] score=0.5910 | CUTE: A Multilingual Dataset for Enhancing Cross-Lingual Kno

--- 有 Reranker (Top-3 of 10) ---
  [1] score=0.0386 | Multilingual NL